In [29]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

# =========================
# 1. CARREGAR DADOS
# =========================
df = pd.read_csv(r"C:\Users\gques\Documents\SPTECH_CODIGOS\ultimo-ano\tcc\backend\tabela_final\part-00000-ca4a34f2-1bd8-485a-bddf-3084ae83539c-c000.csv")

# =========================
# 2. FEATURES E TARGET
# =========================
target = ['delivered_on_time']

X = df.drop(columns= [
    # Vazamento de dados / multicolinearidade
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "interval_code_delivered_carrier",
    "order_delivered_customer_date",
    "interval_code_delivered_customer",
    "order_estimated_delivery_date",
    "shipping_limit_date",
    "is_holiday_in_7_days",
    "is_holiday_in_14_days",
    "had_holiday_7_days_ago",
    "had_holiday_14_days_ago",
    "delivered_on_time",
    "customer_city",
    "seller_city",
    "category_name",
    "review_score"
])
y = df[target]



In [30]:
# Remove registros com valores nulos
X = X.dropna()
y = y.loc[X.index]

# =========================
# 3. SPLIT TEMPORAL
# =========================

split_date = pd.to_datetime("2018-07-01", utc=True)

date_col = pd.to_datetime(
    df.loc[X.index, "order_delivered_carrier_date"],
    utc=True
)

train_mask = date_col < split_date
test_mask = date_col >= split_date

X_train = X.loc[train_mask].copy()
y_train = y.loc[train_mask].copy()

X_test = X.loc[test_mask].copy()
y_test = y.loc[test_mask].copy()

# =========================
# 4. NORMALIZAÇÃO
# =========================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =========================
# 5. MODELO KNN
# =========================

knn = KNeighborsClassifier(n_neighbors=5)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

knn.fit(X_train_scaled, y_train)

# =========================
# 6. PREDIÇÃO
# =========================

y_pred = knn.predict(X_test_scaled)

# =========================
# 7. MÉTRICAS
# =========================
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Treino: (28706, 13)
Teste: (5114, 13)


c:\Users\gques\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neighbors\_classification.py:243: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


Accuracy: 0.8801329683222526

Classification Report:
               precision    recall  f1-score   support

           0       0.50      0.01      0.02       613
           1       0.88      1.00      0.94      4501

    accuracy                           0.88      5114
   macro avg       0.69      0.50      0.48      5114
weighted avg       0.84      0.88      0.83      5114


Confusion Matrix:
 [[   7  606]
 [   7 4494]]


In [31]:
# =========================
# 4. NORMALIZAÇÃO (ESSENCIAL pro KNN)
# =========================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =========================
# 5. MODELO KNN
# =========================

knn = KNeighborsClassifier(n_neighbors=3)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

knn.fit(X_train_scaled, y_train)

# =========================
# 6. PREDIÇÃO
# =========================

y_pred = knn.predict(X_test_scaled)

# =========================
# 7. MÉTRICAS
# =========================

print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Treino: (28706, 13)
Teste: (5114, 13)


c:\Users\gques\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\neighbors\_classification.py:243: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples,), for example using ravel().
  return self._fit(X, y)


Accuracy: 0.879155260070395

Classification Report:
               precision    recall  f1-score   support

           0       0.37      0.01      0.02       613
           1       0.88      1.00      0.94      4501

    accuracy                           0.88      5114
   macro avg       0.62      0.50      0.48      5114
weighted avg       0.82      0.88      0.83      5114


Confusion Matrix:
 [[   7  606]
 [  12 4489]]


In [32]:
# =========================
# 4. NORMALIZAÇÃO (ESSENCIAL pro KNN)
# =========================
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# =========================
# 5. MODELO KNN COM DESEMPATE POR DISTÂNCIA
# =========================
# O parâmetro weights='distance' atribui pesos maiores aos vizinhos mais próximos.
# Isso garante que, em caso de empate na contagem (2 a 2 no K=4), 
# a classe do vizinho mais perto vença.
knn = KNeighborsClassifier(
    n_neighbors=4,
    weights='distance'
)

print("Treino:", X_train.shape)
print("Teste:", X_test.shape)

knn.fit(X_train_scaled, y_train.values.ravel())

# =========================
# 6. PREDIÇÃO
# =========================
y_pred = knn.predict(X_test_scaled)

# =========================
# 7. MÉTRICAS
# =========================
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

Treino: (28706, 13)
Teste: (5114, 13)
Accuracy: 0.8799374266718811

Classification Report:
               precision    recall  f1-score   support

           0       0.47      0.01      0.03       613
           1       0.88      1.00      0.94      4501

    accuracy                           0.88      5114
   macro avg       0.68      0.51      0.48      5114
weighted avg       0.83      0.88      0.83      5114


Confusion Matrix:
 [[   8  605]
 [   9 4492]]
